[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/megusto0/mp-1/blob/master/notebooks/05_fletcher_reeves.ipynb)

# 05. Метод Флетчера-Ривса

Метод сопряженных градиентов для положительно определенной квадратичной функции.

In [ ]:
!pip install -q numpy pandas sympy matplotlib

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

Path("figures").mkdir(exist_ok=True)
Path("tables").mkdir(exist_ok=True)

Q = np.array([[4.0, -2.0], [-2.0, 6.0]])
B = np.array([-8.0, 6.0])
C = 30.0
X0 = np.array([0.0, 0.0])
EPS = 1e-4
X_STAR = np.linalg.solve(Q, -B)
F_STAR = 21.6

def f(point):
    x = np.asarray(point, dtype=float)
    return float(0.5 * x @ Q @ x + B @ x + C)

def grad(point):
    x = np.asarray(point, dtype=float)
    return Q @ x + B

def original_formula(x, y):
    return 2 * (x - 3) ** 2 + 3 * (y + 2) ** 2 - 2 * x * y + 4 * x - 6 * y

def grid():
    x = np.linspace(-1.0, 4.0, 220)
    y = np.linspace(-3.0, 2.0, 220)
    xx, yy = np.meshgrid(x, y)
    return xx, yy, original_formula(xx, yy)

def plot_path(points, title, simplexes=None, out=None):
    xx, yy, zz = grid()
    fig, ax = plt.subplots(figsize=(7, 5), dpi=130)
    contour = ax.contour(xx, yy, zz, levels=25, cmap="viridis")
    ax.clabel(contour, inline=True, fontsize=7)
    if simplexes is not None:
        for simplex in simplexes:
            closed = np.vstack([simplex, simplex[0]])
            ax.plot(closed[:, 0], closed[:, 1], color="#4b5563", alpha=0.35, linewidth=1)
    points = np.asarray(points)
    ax.plot(points[:, 0], points[:, 1], marker="o", linewidth=1.8, markersize=3.5, label="траектория")
    ax.scatter([points[0, 0]], [points[0, 1]], color="#2563eb", s=45, label="x0")
    ax.scatter([X_STAR[0]], [X_STAR[1]], color="red", marker="*", s=120, label="x*")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title(title)
    ax.grid(True, alpha=0.25)
    ax.legend()
    fig.tight_layout()
    if out:
        fig.savefig(out, bbox_inches="tight")
    plt.show()

In [ ]:
def steepest_descent(func, grad_func, q_matrix, x0, tol=1e-4, max_iter=300):
    x = np.asarray(x0, dtype=float)
    history = []
    for iteration in range(max_iter + 1):
        g = grad_func(x)
        norm_g = float(np.linalg.norm(g))
        history.append({"iter": iteration, "x": x.copy(), "f": float(func(x)), "grad_norm": norm_g})
        if norm_g < tol:
            break
        alpha = float((g @ g) / (g @ q_matrix @ g))
        x = x - alpha * g
    return x, float(func(x)), history

def fletcher_reeves(func, grad_func, q_matrix, x0, tol=1e-4, max_iter=300):
    x = np.asarray(x0, dtype=float)
    g = grad_func(x)
    direction = -g
    history = []
    for iteration in range(max_iter + 1):
        norm_g = float(np.linalg.norm(g))
        history.append({"iter": iteration, "x": x.copy(), "f": float(func(x)), "grad_norm": norm_g})
        if norm_g < tol:
            break
        alpha = float(-(g @ direction) / (direction @ q_matrix @ direction))
        x_next = x + alpha * direction
        g_next = grad_func(x_next)
        beta = float((g_next @ g_next) / (g @ g))
        direction = -g_next + beta * direction
        x, g = x_next, g_next
    return x, float(func(x)), history

In [ ]:
x_min, f_min, history = fletcher_reeves(f, grad, Q, X0, tol=EPS)
print("x =", x_min, "f =", f_min, "iterations =", history[-1]["iter"])

rows = []
for item in history:
    x = item["x"]
    rows.append([item["iter"], x[0], x[1], item["f"], item["grad_norm"]])
df = pd.DataFrame(rows, columns=["k", "x_k", "y_k", "f(x_k)", "grad_norm"])
df.to_csv("tables/tab_4_4_fr.csv", index=False)
display(df)

plot_path(np.array([item["x"] for item in history]), "Метод Флетчера-Ривса", out="figures/fig_4_4_fr_trajectory.png")